<a href="https://colab.research.google.com/github/emazarchie-cpu/Emmanuel-Archibong-ML-Intern/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emazarchie-cpu/Emmanuel-Archibong-ML-Intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
Ranking / scoring

My project focuses on ranking content pages by their potential CTR improvement opportunity. The intended user is a content editor or SEO specialist who needs to decide which pages to investigate first. I will compare each page's click-through rate with that of pages in a similar search position tier, while considering the reliability of its impression volume. The output will be a prioritized list for human review, not an automatic decision to rewrite content.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:


The ideal target is whether a content page has genuine potential to gain additional clicks without requiring a better search position. This potential is not directly observed in the starter dataset.

As a measurable proxy, I will calculate `ctr_gap`, defined as a page's CTR minus the average CTR of pages in the same position tier. A negative gap identifies pages with below average CTR relative to their peers. I will treat this as a directional opportunity signal, not as verified ground truth.

The proxy has limitations: CTR varies with search intent, SERP features, query mix, and impression volume. Pages with very few impressions may have unstable CTR estimates, so I will examine volume and consider an impression threshold before interpreting the results.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
My primary evaluation metric will be Precision@K, with K = 20 and K = 50. It will measure the proportion of pages in the top ranked list that meet a predefined CTR opportunity proxy criterion.

I will compare the model with a simple baseline ranking and report the prevalence of relevant pages in the eligible dataset. The evaluation will distinguish proxy based performance from verified business impact: high Precision@K would show that the ranking concentrates pages meeting the chosen proxy definition, not that editing those pages will necessarily increase clicks.

The eligibility rules, opportunity threshold, and baseline must be defined before model evaluation.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:

from pathlib import Path
import pandas as pd

# Find the project data file from common working directories.
relative_path = Path("data/raw/content_refresh_anonymized.csv")
candidates = [
    Path.cwd() / relative_path,
    Path.cwd().parent / relative_path,
    Path.cwd().parent.parent / relative_path,
]

data_path = next((p for p in candidates if p.is_file()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
else:
    # Fallback when running the notebook in Colab without a local clone.
    url = (
        "https://raw.githubusercontent.com/"
        "emazarchie-cpu/Emmanuel-Archibong-ML-Intern/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nRequired columns:")
required = [
    "content_id", "client_id", "ctr",
    "avg_position", "position_tier", "impressions_90d"
]
missing = [col for col in required if col not in df.columns]
print("Missing required columns:", missing)

if missing:
    raise ValueError("Check the CSV path and data schema.")

print("\nUnique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nPosition tiers:")
display(df["position_tier"].value_counts(dropna=False))

print("\nImpression summary:")
display(df["impressions_90d"].describe())

print("\nMissing values in relevant fields:")
display(
    df[required].isna().sum().rename("missing_count").to_frame()
)
# Treat zero average position as missing, not as rank zero.
eligible = df.loc[
    df["avg_position"].gt(0)
    & df["impressions_90d"].ge(250)
    & df["ctr"].notna()
    & df["position_tier"].notna()
].copy()

# Compare each page with the mean CTR of its position tier.
eligible["peer_avg_ctr"] = (
    eligible.groupby("position_tier")["ctr"]
    .transform("mean")
)

eligible["ctr_gap"] = (
    eligible["ctr"] - eligible["peer_avg_ctr"]
)

print("Eligible pages:", len(eligible))
print("Share of dataset eligible:", round(len(eligible) / len(df), 3))

print("\nCTR gap summary:")
display(eligible["ctr_gap"].describe())

print("\nCTR gap by position tier:")
display(
    eligible.groupby("position_tier")["ctr_gap"]
    .agg(["count", "mean", "median"])
    .sort_values("median")
)

print("\nPages with the most negative CTR gaps:")
display(
    eligible.sort_values("ctr_gap")
    [["content_id", "position_tier", "impressions_90d",
      "ctr", "peer_avg_ctr", "ctr_gap"]]
    .head(10)
)

Dataset shape: (30000, 44)

First five rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Required columns:
Missing required columns: []

Unique content IDs: 30000
Unique clients: 32

Position tiers:


,count
position_tier,
page_1,11814
striking,7304
page_3_5,7242
top_3,2321
deep,1319



Impression summary:


...

,ctr_gap
count,1.938600e+04
mean,-4.031760e-18
std,3.150019e-01
min,-3.368365e-01
25%,-1.679327e-01
50%,-7.683651e-02
75%,7.898545e-02
max,5.172067e+00



CTR gap by position tier:


,count,mean,median
position_tier,,,
top_3,496,1.947367e-17,-0.138387
page_1,7811,-4.357885e-17,-0.106837
striking,5234,3.272973e-17,-0.087933
page_3_5,5224,7.587091e-18,-0.061015
deep,621,-2.838131e-18,-0.046232



Pages with the most negative CTR gaps:


,content_id,position_tier,impressions_90d,ctr,peer_avg_ctr,ctr_gap
12552,content_4852b7840db7,page_1,605,0.0,0.336837,-0.336837
21882,content_2dbab51b83c9,page_1,781,0.0,0.336837,-0.336837
21869,content_5da773cd04b2,page_1,320,0.0,0.336837,-0.336837
21905,content_14f02f6b2395,page_1,404,0.0,0.336837,-0.336837
5848,content_4b40bc3fc714,page_1,548,0.0,0.336837,-0.336837
5829,content_603d43d1089d,page_1,632,0.0,0.336837,-0.336837
21800,content_ce1f9e5f2424,page_1,1115,0.0,0.336837,-0.336837
145,content_65629be9dbb5,page_1,1230,0.0,0.336837,-0.336837
29842,content_ce7cb7d2a1ff,page_1,292,0.0,0.336837,-0.336837
29841,content_ef2dd3a85644,page_1,772,0.0,0.336837,-0.336837


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
A fixed rule based on CTR gap and impression volume provides a useful, interpretable baseline. However, CTR opportunities may depend on several interacting factors, including search position, content age, word count, search intent, and content type. Machine learning may help combine these signals to improve page prioritization. I will compare any model against the rule-based baseline using the same predefined evaluation metric rather than assume ML is superior. If the model does not improve prioritization meaningfully, the simpler rule may be preferable. Any conclusions will be limited to the measured proxy and evaluation data.

candidate_features = [
    "ctr",
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "content_type",
    "main_intent",
]

available_features = [
    col for col in candidate_features if col in df.columns
]

print("Candidate signals available:")
print(available_features)

print("\nCandidate signals missing from the dataset:")
print(sorted(set(candidate_features) - set(available_features)))

Candidate signals available:
['ctr', 'impressions_90d', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count', 'content_type', 'main_intent']

Candidate signals missing from the dataset:
[]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.